# Detailed Multi-Recall Pipeline Notebook

这个 notebook 将召回全流程拆成多个步骤，方便逐段调试和观察中间结果。

In [8]:
import os
import pickle

from metrics import metrics_recall
from recall_pipeline import (
    RECALL_METHODS,
    cold_start_items,
    combine_recall_results,
    generate_submission_from_recall_dict,
    get_eval_frames,
    load_artifacts,
    run_embedding_recall,
    run_itemcf_recall,
    run_youtube_usercf_recall,
    youtubednn_u2i_dict,
)
from utils.data_utils import get_user_hist_item_info_dict


## 1. 配置

In [9]:
data_path = './tcdata/'
save_path = './temp_results/'

offline = False
metric_recall = True
use_sample = False
sample_nums = 10000

enable_methods = list(RECALL_METHODS)

weight_dict = {
    'itemcf_sim_itemcf_recall': 1.0,
    'youtubednn_usercf_recall': 0.6,
    'youtubednn_recall': 0.3,
    'embedding_sim_item_recall': 0.2,
    'cold_start_recall': 0.0,
}

print('Config loaded')
print(f'use_sample={use_sample}, sample_nums={sample_nums}, offline={offline}, metric_recall={metric_recall}')
print(f'enable_methods={enable_methods}')


Config loaded
use_sample=False, sample_nums=10000, offline=False, metric_recall=True
enable_methods=['itemcf_sim_itemcf_recall', 'embedding_sim_item_recall', 'youtubednn_recall', 'youtubednn_usercf_recall', 'cold_start_recall']


## 2. 加载基础数据与物料

In [10]:
artifacts = load_artifacts(
    data_path=data_path,
    save_path=save_path,
    offline=offline,
    use_sample=use_sample,
    sample_nums=sample_nums,
)

all_click_df = artifacts['all_click_df']
item_info_df = artifacts['item_info_df']
item_type_dict = artifacts['item_type_dict']
item_words_dict = artifacts['item_words_dict']
item_created_time_dict = artifacts['item_created_time_dict']
item_emb_df = artifacts['item_emb_df']

print(all_click_df.shape)
print(item_info_df.shape)
all_click_df.head()



[Pipeline] Preparing artifacts from data_path=./tcdata/, save_path=./temp_results/

[Pipeline] Loading full click data via get_all_click_df(offline=False)

[Pipeline] Click data ready: rows=1630633, users=250000, items=35380

[Pipeline] Item artifacts ready: item_info=364047, item_emb=364047
(1630633, 9)
(364047, 4)


,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,0,30760,0.343711,4,1,17,1,25,2
1,0,157507,0.343719,4,1,17,1,25,2
2,1,289197,0.343613,4,1,17,1,25,6
3,1,63746,0.343622,4,1,17,1,25,6
4,2,36162,0.343647,4,3,20,1,25,2


## 3. 生成评估集 / 召回输入集

In [11]:
recall_click_df, trn_last_click_df = get_eval_frames(all_click_df, metric_recall)
print(recall_click_df.shape)
if trn_last_click_df is not None:
    print(trn_last_click_df.shape)
recall_click_df.head()



[Pipeline] Metric recall enabled, splitting history clicks and last clicks for offline evaluation

[Pipeline] Evaluation split ready: hist_rows=1390153, last_click_rows=250000
(1390153, 9)
(250000, 9)


,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,0,30760,0.343711,4,1,17,1,25,2
1,1,289197,0.343613,4,1,17,1,25,6
2,2,36162,0.343647,4,3,20,1,25,2
3,3,50644,0.343625,4,3,2,1,25,2
4,4,42567,0.343698,4,1,12,1,16,1


## 4. embedding i2i 召回

In [12]:
emb_i2i_sim, embedding_recall_dict = run_embedding_recall(
    recall_click_df,
    item_created_time_dict,
    item_emb_df,
    save_path,
)

print(len(embedding_recall_dict))
first_user = next(iter(embedding_recall_dict))
print(first_user, embedding_recall_dict[first_user][:10])



[Pipeline] Running embedding i2i recall: faiss_topk=10, sim_item_topk=150, recall_item_num=100


364047it [00:01, 207481.92it/s]
embedding_recall: 100%|██████████| 250000/250000 [00:37<00:00, 6580.36it/s]



[Pipeline] Embedding i2i recall finished for 250000 users
250000
0 [(30955, np.float64(6.453203243907665)), (22024, np.float64(6.3926735858886214)), (28747, np.float64(6.073605821880734)), (22992, np.float64(4.2425874284658205)), (31966, np.float64(4.2305410875943235)), (23500, np.float64(4.22115762878775)), (29258, np.float64(4.199308149818729)), (31072, np.float64(4.196955962821524)), (23759, np.float64(4.181503508306012)), (272143, -100)]


## 5. ItemCF 召回

In [13]:
itemcf_recall_dict = run_itemcf_recall(
    recall_click_df,
    item_created_time_dict,
    emb_i2i_sim,
    save_path,
)

print(len(itemcf_recall_dict))
first_user = next(iter(itemcf_recall_dict))
print(first_user, itemcf_recall_dict[first_user][:10])



[Pipeline] Running ItemCF recall: sim_item_topk=20, recall_item_num=10


itemcf_recall: 100%|██████████| 250000/250000 [45:09<00:00, 92.27it/s]  



[Pipeline] ItemCF recall finished for 250000 users
250000
0 [(180441, np.float64(0.8052261591549373)), (111338, np.float64(0.52979009680246)), (36162, np.float64(0.5118305957849781)), (156279, np.float64(0.4920671403832255)), (3027, np.float64(0.4043291845542898)), (312611, np.float64(0.39297515514391823)), (181296, np.float64(0.39233207828397665)), (57416, np.float64(0.36573127652377907)), (96060, np.float64(0.36479948323798533)), (234481, np.float64(0.3601960113750763))]


## 6. YoutubeDNN u2i 召回

In [14]:
youtube_recall_dict, youtube_user_emb_dict = youtubednn_u2i_dict(
    recall_click_df,
    item_info_df,
    save_path,
)

print(len(youtube_recall_dict))
first_user = next(iter(youtube_recall_dict))
print(first_user, youtube_recall_dict[first_user][:10])



[Pipeline] Running YoutubeDNN u2i recall: topk=20, seq_len=30, words_bucket_num=20, epochs=10


100%|██████████| 250000/250000 [00:13<00:00, 18417.23it/s]



[Pipeline] YoutubeDNN samples ready: train=1077081, test=250000

[Pipeline] YoutubeDNN training started on device=cuda


Epoch 1/10: 100%|██████████| 2104/2104 [00:13<00:00, 152.18it/s]


[Trainer] epoch 1 loss: 11723.7392


...

[Trainer] epoch 9 loss: 9340.8986


Epoch 10/10: 100%|██████████| 2104/2104 [00:16<00:00, 128.07it/s]
/root/TCNews/models/DNN.py:225: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  def get_item_embedding(self, item_ids, item_category_ids=None, item_words_ids=None):


[Trainer] epoch 10 loss: 9258.0892

[Pipeline] Extracting YoutubeDNN embeddings

[Pipeline] Running FAISS retrieval for YoutubeDNN u2i


youtube_u2i_recall: 100%|██████████| 250000/250000 [00:02<00:00, 91468.95it/s]



[Pipeline] YoutubeDNN u2i recall finished for 250000 users
250000
235554 [(111289, 0.8729251623153687), (172730, 0.8712505102157593), (243177, 0.8688144683837891), (210124, 0.8675113320350647), (14519, 0.8657108545303345), (137301, 0.8625897765159607), (73812, 0.8623191118240356), (243182, 0.8621755838394165), (209681, 0.8618584871292114), (38204, 0.8615239858627319)]


## 7. YoutubeDNN usercf 召回

In [15]:
youtube_usercf_recall_dict = run_youtube_usercf_recall(
    recall_click_df,
    youtube_user_emb_dict,
    item_created_time_dict,
    emb_i2i_sim,
    save_path,
)

print(len(youtube_usercf_recall_dict))
first_user = next(iter(youtube_usercf_recall_dict))
print(first_user, youtube_usercf_recall_dict[first_user][:10])



[Pipeline] Running YoutubeDNN usercf recall: faiss_topk=10, sim_user_topk=20, recall_item_num=10


250000it [00:01, 136163.14it/s]
youtube_usercf_recall: 100%|██████████| 250000/250000 [03:52<00:00, 1075.83it/s]



[Pipeline] YoutubeDNN usercf recall finished for 250000 users
250000
0 [(36162, np.float64(3.7828852326449263)), (209122, np.float64(3.7818672077750612)), (272143, -100), (123909, -101), (234698, -102), (336221, -103), (336223, -104), (162655, -105), (168623, -106), (96210, -107)]


## 8. 冷启动召回过滤

In [16]:
all_click_with_info = all_click_df.merge(item_info_df, how='left', on='click_article_id')
user_hist_item_typs_dict, user_hist_item_ids_dict, user_hist_item_words_dict, user_last_item_created_time_dict = get_user_hist_item_info_dict(all_click_with_info)
click_article_ids_set = set(all_click_df['click_article_id'].values)

cold_start_recall_dict = cold_start_items(
    embedding_recall_dict,
    user_hist_item_typs_dict,
    user_hist_item_ids_dict,
    user_hist_item_words_dict,
    user_last_item_created_time_dict,
    item_type_dict,
    item_words_dict,
    item_created_time_dict,
    click_article_ids_set,
    recall_item_num=100,
)

print(len(cold_start_recall_dict))
first_user = next(iter(cold_start_recall_dict))
print(first_user, cold_start_recall_dict[first_user][:10])



[Pipeline] Running cold start filter: candidate_users=250000, recall_item_num=100


cold_start_filter: 100%|██████████| 250000/250000 [00:08<00:00, 30868.62it/s]


[Pipeline] Cold start filter finished for 250000 users
250000
0 [(28747, np.float64(6.073605821880734)), (31966, np.float64(4.2305410875943235)), (29258, np.float64(4.199308149818729)), (31072, np.float64(4.196955962821524))]


## 9. 汇总多路召回结果

In [17]:
user_multi_recall_dict = {
    'itemcf_sim_itemcf_recall': itemcf_recall_dict,
    'embedding_sim_item_recall': embedding_recall_dict,
    'youtubednn_recall': youtube_recall_dict,
    'youtubednn_usercf_recall': youtube_usercf_recall_dict,
    'cold_start_recall': cold_start_recall_dict,
}

for method, recall_result in user_multi_recall_dict.items():
    print(method, len(recall_result))


itemcf_sim_itemcf_recall 250000
embedding_sim_item_recall 250000
youtubednn_recall 250000
youtubednn_usercf_recall 250000
cold_start_recall 250000


## 10. 分路评估

In [18]:
if metric_recall and trn_last_click_df is not None:
    for method, recall_result in user_multi_recall_dict.items():
        if recall_result:
            print(f'\n[{method}]')
            metrics_recall(
                recall_result,
                trn_last_click_df,
                topk=min(50, max(len(items) for items in recall_result.values()))
            )
else:
    print('metric_recall is disabled')



[itemcf_sim_itemcf_recall]
 topk:  10  :  hit_num:  83078 hit_rate:  0.33231 user_num :  250000

[embedding_sim_item_recall]
 topk:  10  :  hit_num:  4593 hit_rate:  0.01837 user_num :  250000
 topk:  20  :  hit_num:  15078 hit_rate:  0.06031 user_num :  250000
 topk:  30  :  hit_num:  23315 hit_rate:  0.09326 user_num :  250000
 topk:  40  :  hit_num:  31977 hit_rate:  0.12791 user_num :  250000
 topk:  50  :  hit_num:  40170 hit_rate:  0.16068 user_num :  250000

[youtubednn_recall]
 topk:  10  :  hit_num:  15397 hit_rate:  0.06159 user_num :  250000
 topk:  20  :  hit_num:  23147 hit_rate:  0.09259 user_num :  250000

[youtubednn_usercf_recall]
 topk:  10  :  hit_num:  25850 hit_rate:  0.1034 user_num :  250000

[cold_start_recall]
 topk:  10  :  hit_num:  0 hit_rate:  0.0 user_num :  250000
 topk:  20  :  hit_num:  0 hit_rate:  0.0 user_num :  250000
 topk:  30  :  hit_num:  0 hit_rate:  0.0 user_num :  250000
 topk:  40  :  hit_num:  0 hit_rate:  0.0 user_num :  250000
 topk:  50

## 11. 多路融合

In [19]:
final_recall_items_dict = combine_recall_results(
    user_multi_recall_dict,
    weight_dict=weight_dict,
    topk=150,
    save_path=save_path,
)

print(len(final_recall_items_dict))
first_user = next(iter(final_recall_items_dict))
print(first_user, final_recall_items_dict[first_user][:10])



[Pipeline] Combining multi-recall results with final_topk=150

[Pipeline] Final merged recall ready for 250000 users
250000
0 [(180441, np.float64(1.0)), (36162, np.float64(0.9407287914456293)), (209122, np.float64(0.5999944863782827)), (111338, np.float64(0.38108448668802386)), (31194, 0.3), (28811, 0.29830014316277126), (156279, np.float64(0.296319540745767)), (31691, 0.29305097559798604), (31295, 0.21852691138385316), (31175, 0.20752006691653127)]


## 12. 读取落盘结果检查

In [20]:
final_path = os.path.join(save_path, 'final_recall_items_dict.pkl')
with open(final_path, 'rb') as f:
    final_recall_items_dict_from_disk = pickle.load(f)

print(final_path)
print(len(final_recall_items_dict_from_disk))


./temp_results/final_recall_items_dict.pkl
250000


## 13. 生成提交文件

In [21]:
tst_recall = generate_submission_from_recall_dict(
    final_recall_items_dict,
    data_path=data_path,
    save_path=save_path,
    topk=5,
    model_name='multi_recall_pipeline',
)

print(tst_recall.shape)
tst_recall.head()



[Pipeline] Preparing submission data from test users

[Pipeline] Submission candidate data ready: test_users=50000, recall_rows=5636756

[Pipeline] Submission file generated in ./temp_results/ with model_name=multi_recall_pipeline
(5636756, 3)


,user_id,click_article_id,pred_score
21085972,200000,237870,1.464846
21085973,200000,195804,0.769027
21085974,200000,194214,0.764077
21085975,200000,194939,0.665238
21085976,200000,194823,0.599225
